In [2]:
from datasets import load_dataset
from google.colab import drive
drive.mount('/content/drive')

data_files = {
    "train": "/content/drive/MyDrive/train.parquet",
    "validation": "/content/drive/MyDrive/validate.parquet",
    "test": "/content/drive/MyDrive/test.parquet"
}

dataset = load_dataset("parquet", data_files=data_files)

print(dataset)

Mounted at /content/drive


Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['tokens', 'tags'],
        num_rows: 14041
    })
    validation: Dataset({
        features: ['tokens', 'tags'],
        num_rows: 3250
    })
    test: Dataset({
        features: ['tokens', 'tags'],
        num_rows: 3453
    })
})


In [3]:
print(dataset["train"][0])

{'tokens': ['EU', 'rejects', 'German', 'call', 'to', 'boycott', 'British', 'lamb', '.'], 'tags': [1, 0, 2, 0, 0, 0, 2, 0, 0]}


In [4]:
label_list = [
    "O",
    "B-PER", "I-PER",
    "B-ORG", "I-ORG",
    "B-LOC", "I-LOC",
    "B-MISC", "I-MISC"
]

label2id = {l: i for i, l in enumerate(label_list)}
id2label = {i: l for i, l in enumerate(label_list)}

print(label_list)

['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC', 'B-MISC', 'I-MISC']


In [5]:
from transformers import AutoTokenizer, AutoModelForTokenClassification, TrainingArguments, Trainer
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [6]:
def tokenize_and_align_labels(example):
    tokenized_inputs = tokenizer(
        example["tokens"],
        truncation=True,
        is_split_into_words=True
    )

    labels = []
    word_ids = tokenized_inputs.word_ids()

    previous_word_idx = None
    label_ids = []

    for word_idx in word_ids:
        if word_idx is None:
            label_ids.append(-100)  # special tokens
        elif word_idx != previous_word_idx:
            label_ids.append(example["tags"][word_idx])
        else:
            label_ids.append(-100)  # subwords ignored

        previous_word_idx = word_idx

    tokenized_inputs["labels"] = label_ids
    return tokenized_inputs

In [7]:
tokenized_dataset = dataset.map(tokenize_and_align_labels, batched=False)

Map:   0%|          | 0/14041 [00:00<?, ? examples/s]

Map:   0%|          | 0/3250 [00:00<?, ? examples/s]

Map:   0%|          | 0/3453 [00:00<?, ? examples/s]

In [8]:
model = AutoModelForTokenClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id
)

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForTokenClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [10]:
from seqeval.metrics import classification_report
training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=2,
    #evaluation_strategy="epoch",
    logging_dir="./logs"
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [11]:
def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_labels = []
    true_predictions = []

    for pred, lab in zip(predictions, labels):
        temp_labels = []
        temp_preds = []

        for p, l in zip(pred, lab):
            if l != -100:
                temp_labels.append(label_list[l])
                temp_preds.append(label_list[p])

        true_labels.append(temp_labels)
        true_predictions.append(temp_preds)

    print(classification_report(true_labels, true_predictions))
    return {}

In [12]:
from transformers import DataCollatorForTokenClassification

data_collator = DataCollatorForTokenClassification(tokenizer)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"].select(range(2000)),
    eval_dataset=tokenized_dataset["validation"].select(range(500)),
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

In [13]:
trainer.train()

Step,Training Loss
500,0.223399


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=500, training_loss=0.2233988037109375, metrics={'train_runtime': 34.5574, 'train_samples_per_second': 115.749, 'train_steps_per_second': 14.469, 'total_flos': 44780892808176.0, 'train_loss': 0.2233988037109375, 'epoch': 2.0})

In [14]:
import numpy as np
trainer.evaluate()

              precision    recall  f1-score   support

         LOC       0.89      0.92      0.91       304
        MISC       0.86      0.75      0.80        95
         ORG       0.96      0.98      0.97       332
         PER       0.94      0.85      0.89       279

   micro avg       0.93      0.91      0.92      1010
   macro avg       0.91      0.88      0.89      1010
weighted avg       0.92      0.91      0.91      1010



{'eval_loss': 0.0962664783000946,
 'eval_runtime': 2.3436,
 'eval_samples_per_second': 213.346,
 'eval_steps_per_second': 26.882,
 'epoch': 2.0}

In [16]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

sentence = "John works at Google in California"

inputs = tokenizer(sentence.split(), return_tensors="pt", is_split_into_words=True)
inputs = {k: v.to(device) for k, v in inputs.items()}

outputs = model(**inputs)
predictions = outputs.logits.argmax(dim=2)

predicted_labels = [id2label[p.item()] for p in predictions[0]]

print(list(zip(sentence.split(), predicted_labels)))

[('John', 'O'), ('works', 'B-ORG'), ('at', 'O'), ('Google', 'O'), ('in', 'B-PER'), ('California', 'O')]




POS Tagging vs Chunking
POS Tagging → identifies word type (Noun, Verb)
Chunking → identifies phrases (Noun Phrase, Verb Phrase)

Example:

POS: John → NOUN
Chunk: John → B-NP (Beginning of Noun Phrase

Challenges:
Handling subwords
Label alignment
Observations:
DistilBERT works well for small datasets
Token classification requires careful preprocessing

In [ ]:
import nbformat
from google.colab import _message

# Get current notebook from Colab memory
nb_dict = _message.blocking_request('get_ipynb')['ipynb']

# Convert to notebook format
nb = nbformat.from_dict(nb_dict)

# Remove widget metadata (fix error)
if "widgets" in nb.metadata:
    del nb.metadata["widgets"]

# Save cleaned notebook
with open("/content/clean_notebook.ipynb", "w", encoding="utf-8") as f:
    nbformat.write(nb, f)

print("Clean notebook saved!")

In [20]:
!ls

drive  results	sample_data
